# Handoffs

Handoffs allow an agent to delegate tasks to another agent. This is particularly useful in scenarios where different agents specialize in distinct areas. For example, a customer support app might have agents that each specifically handle tasks like order status, refunds, FAQs, etc.

Handoffs are represented as tools to the LLM. So if there's a handoff to an agent named Refund Agent, the tool would be called transfer_to_refund_agent.

[Learning Reference](https://openai.github.io/openai-agents-python/handoffs/)

## Install openai-agents SDK

In [ ]:
!pip install -Uq openai-agents

## Make your Notebook capable of running asynchronous functions.
Both Jupyter notebooks and Python’s asyncio library utilize event loops, but they serve different purposes and can sometimes interfere with each other.

The nest_asyncio library allows the existing event loop to accept nested event loops, enabling asyncio code to run within environments that already have an event loop, such as Jupyter notebooks.

In summary, both Jupyter notebooks and Python’s asyncio library utilize event loops to manage asynchronous operations. When working within Jupyter notebooks, it’s essential to be aware of the existing event loop to effectively run asyncio code without conflicts.

In [1]:
import nest_asyncio
nest_asyncio.apply()

## Config

In [2]:
from pydantic import BaseModel
from agents import (
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    RunConfig
)
import os


In [19]:
API_KEY = os.environ.get("AIHUBMIX_API_KEY")
BASE_URL = os.environ.get("AIHUBMIX_BASE_URL")

# Check if the API key is present; if not, raise an error
if not API_KEY:
    raise ValueError("GEMINI_API_KEY is not set. Please ensure it is defined in your .env file.")

#Reference: https://ai.google.dev/gemini-api/docs/openai
external_client = AsyncOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

model = OpenAIChatCompletionsModel(
    model="gpt-5-nano",
    openai_client=external_client
)

config = RunConfig(
    model=model,
    model_provider=external_client,
    tracing_disabled=True
)

# Creating a handoff

All agents have a handoffs param, which can either take an Agent directly, or a Handoff object that customizes the Handoff.

In [5]:
import asyncio
from agents import Agent, handoff, Runner

### 1. Basic Usage

In [20]:
chinese_agent = Agent(
    name="Chinese agent",
    instructions="You only speak Chinese."
)

urdu_agent = Agent(
    name="Urdu agent",
    instructions="You only speak Urdu"
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request. default chinese",
    handoffs=[urdu_agent, chinese_agent],
)


async def main(input: str):
    result = await Runner.run(triage_agent, input=input, run_config=config)
    print(result.final_output)


In [22]:
asyncio.run(main("السلام عليكم"))

وعلیکم السلام۔ میں اردو میں آپ کی کس طرح مدد کر سکتا ہوں؟ کوئی مخصوص سوال یا موضوع جس پر بات کرنا چاہیں؟


In [23]:
asyncio.run(main("Hi"))

你好！很高兴见到你。有什么需要我帮忙的吗？无论是问答、翻译、写作、编程、学习资料，还是其他方面，都可以告诉我。你现在想聊点什么？


### 2. Customizing handoffs via the handoff() function

In [24]:
from agents import Agent, handoff, RunContextWrapper

urdu_agent = Agent(
    name="Urdu agent",
    instructions="You only speak Urdu."
)

english_agent = Agent(
    name="English agent",
    instructions="You only speak English"
)

def on_handoff(agent: Agent, ctx: RunContextWrapper[None]):
    agent_name = agent.name
    print("--------------------------------")
    print(f"Handing off to {agent_name}...")
    print("--------------------------------")

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[
            handoff(urdu_agent, on_handoff=lambda ctx: on_handoff(urdu_agent, ctx)),
            handoff(english_agent, on_handoff=lambda ctx: on_handoff(english_agent, ctx))
    ],
)


async def main(input: str):
    result = await Runner.run(triage_agent, input=input, run_config=config)
    print(result.final_output)


In [28]:
asyncio.run(main("السلام عليكم"))

--------------------------------
Handing off to Urdu agent...
--------------------------------
وعلیکم السلام ورحمۃ اللہ وبرکاتہ۔ میں آپ کی کس طرح مدد کروں؟


In [27]:
asyncio.run(main("hello"))

--------------------------------
Handing off to English agent...
--------------------------------
Hello! How can I assist you today? I can answer questions, explain concepts, draft emails or documents, help with brainstorming, summarize articles, translate text, or just chat. What would you like to do?


### Self Assignment

Implement the following with HandOffs Pattern:
- Handoff Custom Inputs
- Set an input_filter
- Share info about handoffs in your agents
- Implement Streaming With HandOff

Here are some research references to get started:
- https://openai.github.io/openai-agents-python/handoffs/
- https://github.com/openai/openai-agents-python/blob/main/examples/handoffs/message_filter_streaming.py
- https://github.com/openai/openai-agents-python/blob/main/examples/handoffs/message_filter.py